# Indexing

In [1]:
from haystack import Pipeline, Document
from milvus_haystack import MilvusDocumentStore
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter

# file_names = ["Lec7 LLM Fine-Tuning.pdf"]
file_names = ["Project Management Requirements Handbook.pdf"]

""" Connect to Milvus Lite DB; create new collection and drop old if exists """
document_store = MilvusDocumentStore(
    collection_name="HaystackCollection",
    collection_description="Getting started with Milvus integration",
    connection_args={"uri": "./milvus.db"}, # Milvus Lite
    #connection_args={"uri": "http://localhost:19530"}, # Milvus standalone Docker service
    index_params={
        "metric_type": "L2",
        "index_type": "FLAT",
    },
    drop_old=True,
)

In [3]:
pipe = Pipeline()

pipe.add_component("converter", PyPDFToDocument(extraction_mode="layout"))
pipe.add_component("cleaner", DocumentCleaner())
pipe.add_component("splitter", DocumentSplitter(split_by="word", split_length=100, split_overlap=10, split_threshold=50))
pipe.add_component("embedder", SentenceTransformersDocumentEmbedder())
pipe.add_component("writer", DocumentWriter(document_store=document_store))

pipe.connect("converter", "cleaner")
pipe.connect("cleaner", "splitter")
pipe.connect("splitter", "embedder")
pipe.connect("embedder", "writer")

🚅 Components
  - converter: PyPDFToDocument
  - cleaner: DocumentCleaner
  - splitter: DocumentSplitter
  - embedder: SentenceTransformersDocumentEmbedder
  - writer: DocumentWriter
🛤️ Connections
  - converter.documents -> cleaner.documents (List[Document])
  - cleaner.documents -> splitter.documents (List[Document])
  - splitter.documents -> embedder.documents (List[Document])
  - embedder.documents -> writer.documents (List[Document])

In [4]:
results = pipe.run({"converter": {"sources": file_names}}, include_outputs_from={"converter", "cleaner", "splitter"})

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Document 897ba1906797de32cfea2c683fb590d1733ea0be2bf7bd7d00541923937d6f9a has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document 4b2a525a6a7bb7ed28badb0ff22fdb31717c2ef1ae95f90d8b3ccd3115ec31db has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document af6d2261b0b0b92ec74d222bb3665b639be914f0ca45708aadd4af6572c09aa0 has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document 3099f7000d651c6613869c8dd626dbe67078a1c186505f9c3bcc1c74891821e7 has metadata fields with unsupported types: ['_split_overlap']. Supported types refer to Pymilvus DataType. The values of these fields will be discarded.
Document 4df66944e775c527921be578fd4f590f3ceb106debb45d10bdb84cc830f7fce4 has metadata f

In [5]:
results.keys()

dict_keys(['writer', 'converter', 'cleaner', 'splitter'])

In [ ]:
print(results["converter"]["documents"][0].content)

In [ ]:
print(results["cleaner"]["documents"][0].content)

In [6]:
document_store.count_documents()

40

In [9]:
for idx in range(10):
    print(f"Doc {idx}: {document_store.filter_documents()[idx].content}\n")

Doc 0: with the 1st project, then we willput the student in the pool for consideration ■ Ifit’sa really urgent request from a client then Beam Data may override the rules by selecting the student who has specific skills required for the project - E.g., ifthere’s only one student knows Elasticsearch database then we may choose that student for help ○ How ready isthe student for jobs? ■ Ifthe student has not accomplished much on building the resume and portfolio, then we’d favor the student less because the faculty may want the student to focus on job search ● Exceptions will be 

Doc 1: project ● Overall contribution ● Documentation Peer comments: ● Eg. Improve communication skills/ Should practise time management skills etc. Your 1st project review will affect the decision your 2nd project selection 6.2 Manager review (40%) ● Team lead review ● PM review ● Instructors
6.3 Client Review (team level) (30%) ● We ask clients to review the entire project team when finished. Allthe reviews a

# Retrieval

In [10]:
from milvus_haystack.milvus_embedding_retriever import MilvusEmbeddingRetriever
from haystack.components.embedders import SentenceTransformersTextEmbedder

In [12]:
retrieval_pipe = Pipeline()
retrieval_pipe.add_component("query_embedder", SentenceTransformersTextEmbedder())
retrieval_pipe.add_component("retriever", MilvusEmbeddingRetriever(document_store=document_store, top_k=3))

retrieval_pipe.connect("query_embedder", "retriever")

🚅 Components
  - query_embedder: SentenceTransformersTextEmbedder
  - retriever: MilvusEmbeddingRetriever
🛤️ Connections
  - query_embedder.embedding -> retriever.query_embedding (List[float])

In [31]:
question = "How long does a project usually take?"
question = "How many projects can I do?"
question = "What are the primary rules I must follow to work for BeamData?"

results = retrieval_pipe.run({"query_embedder": {"text": question}})
for doc in results["retriever"]["documents"]:
    print(doc.content)
    print("-"*10)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Project Manager (L2) and the BeamData faculty member. We’ll assign students to 2nd project based on the best overall fit and the project needs (eg. 1st project performance, GPA, soft skills,time schedule) 6weclouddata.com Ifyou want to wait for the “perfect” 2nd project it’sfine. But we encourage you to take
the one that has immediate openings, which helps you fillthe gap during your job
search. 1.2 How many projects can a student work on? Each student can work on 1-2 projects under the titleof Data Science Consultant.
There willbe special occasions that Beam Data may proactively bring some students to
work on additional projects to 
----------
by the client or BeamData before starting the project. ● Things to avoid ○ Don’t share client information ○ Don’t disclose listof clients of beam data ○ Don’t publish any client data ○ Don’t use Tableau/PowerBI (Free Version) forclient projects. 7.2 What information can be shared on portfolio/resume/linkedin Always communicate with the PM or th

# RAG

In [14]:
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator

In [23]:
prompt_template = """Answer the following query based on the provided context. If the context does
                     not include an answer, reply with 'I don't know'.\n
                     Query: {{query}}
                     Documents:
                     {% for doc in documents %}
                        {{ doc.content }}
                     {% endfor %}
                     Answer: 
                  """

In [24]:
rag_pipe = Pipeline()

rag_pipe.add_component("query_embedder", SentenceTransformersTextEmbedder())
rag_pipe.add_component("retriever", MilvusEmbeddingRetriever(document_store = document_store, top_k=3))
rag_pipe.add_component("prompt_builder", PromptBuilder(template=prompt_template))
rag_pipe.add_component("generator", OpenAIGenerator(generation_kwargs={"temperature": 0.7, "max_tokens": 500}))

rag_pipe.connect("query_embedder", "retriever")
rag_pipe.connect("retriever", "prompt_builder")
rag_pipe.connect("prompt_builder", "generator")

🚅 Components
  - query_embedder: SentenceTransformersTextEmbedder
  - retriever: MilvusEmbeddingRetriever
  - prompt_builder: PromptBuilder
  - generator: OpenAIGenerator
🛤️ Connections
  - query_embedder.embedding -> retriever.query_embedding (List[float])
  - retriever.documents -> prompt_builder.documents (List[Document])
  - prompt_builder.prompt -> generator.prompt (str)

In [30]:
question = "How long does a project usually take?"
question = "How many projects can I do?"
question = "What are the primary rules I must follow to work for BeamData?"

results = rag_pipe.run({"query_embedder": {"text": question},
                        "prompt_builder": {"query": question}}, include_outputs_from={"retriever"})
print(results["generator"]["replies"][0])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

The primary rules to follow when working for BeamData include:

1. Avoid sharing client information or disclosing the list of BeamData clients.
2. Do not publish any client data.
3. Do not use Tableau/PowerBI (Free Version) for client projects.
4. Always communicate with the Project Manager (PM) or the client regarding the information you want to share on your public profile.
5. Follow the escalation procedure: report issues to the project lead first, and if unresolved, escalate to the project manager.
